# KAT-VAD v5 — **Stage 0b**: Amendment 2 re-extract + re-read, then check L (lag × centre shift)

Rules: `core/docs/v5/PREREG_V5.md` **§1.2 (Amendment 2)** and **§1.3 (check L, L-1…L-3)** — both written after the first
read-out (INVALID: 15 dev clips failed `g1_pixel` because v2's all-frame pixel gate was re-taken on `s1[::3]`).

| step | what | GPU / I/O |
|---|---|---|
| 2 | re-extract **only** the 15 `g1_pixel` clips with the amended gate (`--retry g1_pixel`) | GPU, streams only the tar groups that hold them |
| 3 | coverage: expect 0 failed, 569 maps, 1,129 grids | — |
| 4 | Stage 0 read-out again (§1 + Amendments 1–2) | CPU |
| 5 | check L: V7 change at row lags 1 / 3 / 6 × centre shift 0 / 2 / 3 rows from the stored grids (printed, never gating; L-3 picks Stage 1's window) | CPU, no tar |

Same Drive layout and code upload as `stage0_dota_cap_tokens.ipynb` (re-upload `kat-vad/` first — this needs the
`--retry` flag and the `lag` command).

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'      # read-only here: v2 caches
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload the v5 code there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'V5_LAG_ROWS'), 'this upload predates Amendment 2 / check L -- re-upload the code'

ENC      = constants.V5_STAGE0_ENCODER
GROUPS   = ['1-10', '11', '12-42', '43', '44-62']
BATCH_GB = constants.MMAU_STREAM_BATCH_GB
RAW      = constants.DATA_ROOT / 'MMAU' / 'raw'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
V2_CACHE = constants.VIDEO_CACHE_DIR / ENC / f'{constants.DOTA_CAP_DATASET}_s1_squash'
DCAP     = constants.OUTPUT_ROOT / 'REPORTS' / 'dota_cap'
ALIGN    = DCAP / 'dota_cap_alignment.json'
FINAL    = DCAP / 'dota_cap_readout.json'
OUT      = constants.OUTPUT_ROOT / constants.V5_OUTPUT_DIRNAME / constants.V5_STAGE0_DIRNAME
GRID_DIR = Path(DRIVE_V2) / 'cache' / constants.V5_GRID_CACHE_DIRNAME / ENC / constants.V5_GRID_DATASET_TAG
LOCAL    = Path('/content/v5s0')
for p in (OUT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name, path in (('DoTA s1', DOTA_S1), ('V2-S DoTA-CAP s1', V2_CACHE), ('alignment', ALIGN),
                   ('final ids', FINAL), ('CAP annotation', RAW / 'cap_text_annotations.xls'),
                   ('CAP chunks', RAW / 'CAP-DATA_chunks')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:18s} {path}   OK')
print('grid cache ->', GRID_DIR)
print('maps/report ->', OUT)

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu xlrd
python -c "import av, einops, faiss, xlrd; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 2 needs one'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time


def stream(cmd, name):
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print(''.join(tail))
        raise RuntimeError(f'{name} exited {proc.returncode}')

## 2. Re-extract the `g1_pixel` clips (Amendment 2) — GPU

Each failed clip is redone in its own group's report; only its CAP video is kept from the stream, and the tool stops
reading the group's tar once that video is closed.

In [ ]:
retry = {}
for p in sorted(OUT.glob('extract_*.json')):
    group = p.stem.removeprefix('extract_')
    for dota_id, e in json.loads(p.read_text()).items():
        if e['reason'] == 'g1_pixel':
            retry.setdefault(group, []).append((dota_id, e['cap_id']))
print({g: len(v) for g, v in retry.items()}, '-> total', sum(map(len, retry.values())), '(expect 15)')

free_gb = shutil.disk_usage('/content').free / 1e9
assert free_gb >= 3 * BATCH_GB, f'{free_gb:.0f} GB free on /content, need {3 * BATCH_GB:.0f}'
for g, clips in retry.items():
    ids_file = LOCAL / f'retry_cap_ids_{g}.txt'
    ids_file.write_text(''.join(f'{c}\n' for c in sorted({cap for _, cap in clips})))
    parts = sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
    assert parts, f'no parts for group {g}'
    t0 = time.time()
    stream([sys.executable, '-m', 'core.tools.v5_stage0', 'extract',
            '--alignment', ALIGN, '--final-readout', FINAL, '--parts', *parts,
            '--work-dir', LOCAL / 'work' / g, '--dota-clip-dir', DOTA_S1, '--v2-cache', V2_CACHE,
            '--report', OUT / f'extract_{g}.json', '--out-dir', OUT, '--grid-dir', GRID_DIR,
            '--cap-ids-file', ids_file, '--batch-size', '32', '--max-batch-gb', str(BATCH_GB),
            '--device', 'auto', '--retry', 'g1_pixel'], 'v5_stage0 extract --retry')
    print(f'group {g}: {(time.time() - t0) / 60:.0f} min')

## 3. Coverage check

In [ ]:
from collections import Counter

final_ids = set(json.loads(FINAL.read_text())['ids'])
report = {}
for p in sorted(OUT.glob('extract_*.json')):
    report |= json.loads(p.read_text())
print('clips reported:', len(report), 'of', len(final_ids))
print('reasons:', Counter(e['reason'] for e in report.values()))
print('dev maps:', len(list((OUT / 'maps').glob('*.npz'))), '(expect 569)')
print('grids:', len(list(GRID_DIR.glob('*.npy'))), '(expect 1129)')
redone = [report[d] for g in retry.values() for d, _ in g]
print('redone pixel cos (rows): mean', [round(e.get('pixel_mean_cos', float('nan')), 4) for e in redone])
print('                          min ', [round(e.get('pixel_min_cos', float('nan')), 4) for e in redone])
bad = {k: v['reason'] for k, v in report.items() if v['reason'] != 'ok'}
print('failed:', bad)

## 4. Stage 0 read-out again (§1 + Amendments 1–2) — CPU

The decision is mechanical. Expected before the run (§1.2): **PARTIAL** (G-B failed on 514 clips with CIs on 0).

In [ ]:
%%bash
set -e
mkdir -p /content/v5s0/dota && cd /content/v5s0/dota
[ -d annotations ] || { curl -sSL -o ann.zip https://github.com/MoonBlvd/Detection-of-Traffic-Anomaly/raw/master/dataset/DoTA_annotations.zip && unzip -q ann.zip && rm ann.zip; }
ls annotations | wc -l

In [ ]:
META = constants.DATA_ROOT / 'DoTA' / 'metadata_val.json'
if not META.exists():
    META = LOCAL / 'dota' / 'metadata_val.json'
    subprocess.run(['curl', '-sSL', '-o', str(META),
                    'https://raw.githubusercontent.com/MoonBlvd/Detection-of-Traffic-Anomaly/master/dataset/metadata_val.json'],
                   check=True)
SPLIT = LOCAL / 'dota' / 'val_ids.txt'
SPLIT.write_text(''.join(f'{v}\n' for v in json.loads(META.read_text())))
stream([sys.executable, '-m', 'core.tools.v5_stage0', 'readout',
        '--reports', *sorted(OUT.glob('extract_*.json')), '--maps-dir', OUT / 'maps',
        '--annotations-dir', LOCAL / 'dota' / 'annotations', '--metadata', META,
        '--split-file', SPLIT, '--out-dir', OUT / 'REPORTS'], 'v5_stage0 readout')
print((OUT / 'REPORTS' / 'stage0_readout.md').read_text())

## 5. Check L — lag × centre shift (§1.3, printed only) — CPU

Reads the fp16 V7 grids already on Drive (dev clips only; no DoTA-CAP-eval label is read).

* **lag** ℓ ∈ {1, 3, 6} rows: change against the window ℓ rows back; ℓ = 6 (18 frames) shares no frame. Lag 1 must
  reproduce Stage 0's V7 (`max |Δ|` vs stored fp32 `d7` printed).
* **shift** s ∈ {0, 2, 3} rows: step k's box/label is read against grid row k + s, i.e. a window centred at
  t − 7.5 (causal, as Stage 0), t − 1.5, t + 1.5 frames — the proposal's centred window, without re-extraction.

The md's bold line is L-3's mechanical verdict: **centred** (Stage 1 re-extracts Nexar centred, row 1 retrained) or
**causal** (v2-comparable).

In [ ]:
stream([sys.executable, '-m', 'core.tools.v5_stage0', 'lag',
        '--reports', *sorted(OUT.glob('extract_*.json')), '--maps-dir', OUT / 'maps',
        '--annotations-dir', LOCAL / 'dota' / 'annotations', '--metadata', META,
        '--split-file', SPLIT, '--out-dir', OUT / 'REPORTS', '--grid-dir', GRID_DIR], 'v5_stage0 lag')
print((OUT / 'REPORTS' / 'stage0_lag.md').read_text())

## 6. Record

Paste **both** `REPORTS/stage0_readout.md` and `REPORTS/stage0_lag.md` back to Claude, and bring
`outputs/v5/v5_stage0/{REPORTS,extract_*.json}` to the repo.

In [ ]:
drive.flush_and_unmount()
print('Drive flushed')